In [ ]:
# notebooks/week3_eda.ipynb
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

RAW_PATH = Path("../data/raw/variant_15/2026-05-21_13-20-28.json")
NORMALIZED_DIR = Path("../data/normalized")
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

with open(RAW_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

def normalize_earthquakes(data):
    features = data.get('features', [])
    
    records = []
    for eq in features:
        props = eq.get('properties', {})
        geometry = eq.get('geometry', {})
        coords = geometry.get('coordinates', [None, None, None])
        
        record = {
            'event_id': eq.get('id'),
            'mag': props.get('mag'),
            'mag_type': props.get('magType'),
            'place': props.get('place'),
            'time': pd.to_datetime(props.get('time'), unit='ms'),
            'updated': pd.to_datetime(props.get('updated'), unit='ms'),
            'url': props.get('url'),
            'felt': props.get('felt'),
            'cdi': props.get('cdi'),
            'mmi': props.get('mmi'),
            'alert': props.get('alert'),
            'status': props.get('status'),
            'tsunami': props.get('tsunami'),
            'sig': props.get('sig'),
            'net': props.get('net'),
            'nst': props.get('nst'),
            'dmin': props.get('dmin'),
            'rms': props.get('rms'),
            'gap': props.get('gap'),
            'latitude': coords[1],
            'longitude': coords[0],
            'depth_km': coords[2],
            'event_type': props.get('type'),
            'title': props.get('title')
        }
        records.append(record)
    
    return pd.DataFrame(records)

df = normalize_earthquakes(raw_data)

df['mag'] = pd.to_numeric(df['mag'], errors='coerce')
df['depth_km'] = pd.to_numeric(df['depth_km'], errors='coerce')
df['felt'] = pd.to_numeric(df['felt'], errors='coerce')
df['sig'] = pd.to_numeric(df['sig'], errors='coerce')
df['nst'] = pd.to_numeric(df['nst'], errors='coerce')
df['dmin'] = pd.to_numeric(df['dmin'], errors='coerce')
df['rms'] = pd.to_numeric(df['rms'], errors='coerce')
df['gap'] = pd.to_numeric(df['gap'], errors='coerce')

df['time_year'] = df['time'].dt.year
df['time_month'] = df['time'].dt.month
df['time_day'] = df['time'].dt.day
df['time_hour'] = df['time'].dt.hour

df['felt'].fillna(0, inplace=True)
df['tsunami'].fillna(0, inplace=True)

rename_map = {
    'mag': 'magnitude',
    'mag_type': 'magnitude_type',
    'sig': 'significance',
    'nst': 'num_stations',
    'dmin': 'min_distance_deg',
    'gap': 'azimuthal_gap'
}
df = df.rename(columns=rename_map)

output_filename = RAW_PATH.stem + ".csv"
output_path = NORMALIZED_DIR / output_filename
df.to_csv(output_path, index=False)

C:\Users\dizy\AppData\Local\Temp\ipykernel_14016\3206540178.py:69: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['felt'].fillna(0, inplace=True)
C:\Users\dizy\AppData\Local\Temp\ipykernel_14016\3206540178.py:70: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inpl